# WZML-X en Google Colab

Este notebook usa **udocker** (no FREEROOT ni el daemon de Docker) para ejecutar en espacio de usuario la misma imagen base que emplea el `Dockerfile`: `mysterysd/wzmlx:wzadv`. Así se reutilizan el entorno `/wzvenv`, los binarios nativos y el binding de MEGA. El repositorio se monta dentro del contenedor y se inicia con `python -m bot`, sin ejecutar `start.sh` ni `update.py`.

> **Importante:** Colab es efímero y puede desconectarse. Úsalo para pruebas, no como alojamiento 24/7. Activa el acceso a Internet; el pull de la imagen puede tardar y ocupar varios GB. `DATABASE_URL` es opcional, pero necesaria para conservar ajustes entre sesiones.

Los secretos se solicitan sin eco o se leen de Colab Secrets; `config.py` solo queda en el disco temporal. udocker/PRoot **no proporciona aislamiento fuerte**: ejecuta solo la imagen de confianza del proyecto. La carpeta de descargas desaparece al terminar el runtime.

In [ ]:
# Repositorio/branch a ejecutar. Cambia estos valores si usas un fork.
REPO_URL = "https://github.com/leomarmol03/WZML-X.git"
BRANCH = "arena/d3f87987-wzml-x"  # Branch de este PR; usa "wzv3" después de integrarlo.

from pathlib import Path
import os
import subprocess

ROOT = Path("/content/WZML-X")
if not (ROOT / "bot").is_dir():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(
            f"{ROOT} existe pero no parece una copia de WZML-X; elige otra ruta "
            "o vacíala manualmente."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(ROOT)],
        check=True,
    )
os.chdir(ROOT)
print(f"Repositorio listo: {ROOT} (branch {BRANCH})")

## 1. Preparar udocker

udocker ejecuta contenedores sin Docker daemon ni privilegios reales. Se usa el modo P2 de PRoot, más conservador que el modo acelerado P1 en kernels recientes. La celda siguiente instala udocker, descarga la imagen base del proyecto y monta este repositorio en `/usr/src/app`.

No se instala FREEROOT: no hace falta para este flujo y se prefiere el runtime documentado de udocker.

In [ ]:
import shutil
import subprocess

# Solo hacen falta herramientas del runtime anfitrión para clonar, instalar
# udocker y abrir el túnel. Aria2, qBittorrent, ffmpeg, rclone y Java viven
# dentro de la imagen del proyecto.
required_host_tools = ("git", "curl", "tar", "dpkg")
missing = [name for name in required_host_tools if not shutil.which(name)]
if missing:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        [
            "apt-get", "install", "-y", "-qq", "--no-install-recommends",
            "ca-certificates", *missing,
        ],
        check=True,
    )
missing = [name for name in required_host_tools if not shutil.which(name)]
if missing:
    raise RuntimeError("Faltan herramientas en Colab: " + ", ".join(missing))
print("Herramientas anfitrión verificadas:", ", ".join(required_host_tools))

## 2. Instalar udocker, la imagen y dependencias del bot

La imagen base procede del `FROM mysterysd/wzmlx:wzadv` del Dockerfile. El notebook instala `requirements.txt` **dentro de esa imagen**, en `/wzvenv`, igual que el Dockerfile del proyecto. También comprueba que los binarios y el binding nativo de MEGA se puedan importar.

Si el pull de Docker Hub falla por red o rate limit, vuelve a ejecutar esta celda más tarde o revisa el log de la celda.

In [ ]:
import compileall
import os
import shutil
import subprocess
import sys

previous_bot = globals().get("bot_process")
if previous_bot is not None and previous_bot.poll() is None:
    raise RuntimeError("Detén el bot antes de volver a preparar udocker.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "udocker==1.3.17"],
    check=True,
)
UDOCKER = shutil.which("udocker")
if not UDOCKER:
    raise RuntimeError("No se encontró udocker tras instalarlo con pip.")

# Conserva imagen, contenedor y PRoot solo durante esta sesión temporal.
UDOCKER_HOME = Path("/content/.udocker")
os.environ["UDOCKER_DIR"] = str(UDOCKER_HOME)
UDOCKER_HOME.mkdir(parents=True, exist_ok=True)

subprocess.run([UDOCKER, "install"], check=True)
IMAGE = "mysterysd/wzmlx:wzadv"
CONTAINER_NAME = "wzmlx-colab"
existing = subprocess.run(
    [UDOCKER, "ps", "-m", "-s"], capture_output=True, text=True, check=True
)
if CONTAINER_NAME in existing.stdout:
    subprocess.run([UDOCKER, "rm", "-f", CONTAINER_NAME], check=True)
subprocess.run([UDOCKER, "pull", IMAGE], check=True)
subprocess.run([UDOCKER, "create", f"--name={CONTAINER_NAME}", IMAGE], check=True)
subprocess.run([UDOCKER, "setup", "--execmode=P2", CONTAINER_NAME], check=True)

install_in_image = r"""
set -eu
test -x /wzvenv/bin/python
command -v uv
uv pip install --python /wzvenv/bin/python --no-cache-dir -r /usr/src/app/requirements.txt
uv pip check --python /wzvenv/bin/python
/wzvenv/bin/python -m compileall -q /usr/src/app/bot /usr/src/app/web
/wzvenv/bin/python -c 'from bot.core.config_manager import BinConfig; from bot.helper.ext_utils.mega_sdk import MEGA_SDK_AVAILABLE; print("System binaries:", BinConfig.ARIA2_NAME, BinConfig.QBIT_NAME, BinConfig.FFMPEG_NAME, BinConfig.RCLONE_NAME); print("Compatible MEGA SDK:", MEGA_SDK_AVAILABLE)'
""".strip()
subprocess.run(
    [
        UDOCKER, "run", "--user=root",
        f"--volume={ROOT}:/usr/src/app",
        "--workdir=/usr/src/app",
        "--entrypoint=/bin/bash",
        CONTAINER_NAME, "-c", install_in_image,
    ],
    check=True,
)
if not compileall.compile_dir(str(ROOT / "bot"), quiet=1) or not compileall.compile_dir(
    str(ROOT / "web"), quiet=1
):
    raise RuntimeError("Hay un error de sintaxis en el código del bot/web")
print("udocker, imagen y dependencias del bot: listos.")

## 3. Configuración privada

Puedes crear secretos en **Colab → Secrets** con los nombres indicados. Si no existen, el notebook los solicita de forma oculta. `OWNER_ID` es tu ID numérico de Telegram; obtén `TELEGRAM_API` y `TELEGRAM_HASH` en `my.telegram.org`. La URI de MongoDB se puede dejar vacía, aunque sin ella la configuración no sobrevive al runtime.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def read_secret(name, prompt, optional=False):
    value = ""
    if userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            pass
    if not value:
        value = getpass(prompt)
    value = str(value).strip()
    if not value and not optional:
        raise ValueError(f"{name} es obligatorio")
    return value

BOT_TOKEN = read_secret("BOT_TOKEN", "BOT_TOKEN de @BotFather: ")
OWNER_ID = int(read_secret("OWNER_ID", "Tu OWNER_ID (número de Telegram): "))
TELEGRAM_API = int(read_secret("TELEGRAM_API", "TELEGRAM_API (ID numérico de my.telegram.org): "))
TELEGRAM_HASH = read_secret("TELEGRAM_HASH", "TELEGRAM_HASH de my.telegram.org: ")
DATABASE_URL = read_secret(
    "DATABASE_URL", "MongoDB Atlas URI (opcional, Enter para omitir): ", optional=True
)
if OWNER_ID <= 0 or TELEGRAM_API <= 0 or ":" not in BOT_TOKEN:
    raise ValueError("OWNER_ID, TELEGRAM_API o BOT_TOKEN no tienen un formato válido")

# Aria2/qBittorrent siguen habilitados; servicios pesados se dejan apagados.
config_values = {
    "BOT_TOKEN": BOT_TOKEN,
    "OWNER_ID": OWNER_ID,
    "TELEGRAM_API": TELEGRAM_API,
    "TELEGRAM_HASH": TELEGRAM_HASH,
    "DATABASE_URL": DATABASE_URL,
    "BASE_URL": "",
    "DISABLE_MEGA": False,
    "DISABLE_TORRENTS": False,
    "DISABLE_STREAM": False,
    "DISABLE_JD": True,
    "DISABLE_NZB": True,
    "BOT_MAX_TASKS": 2,
    "USER_MAX_TASKS": 2,
}
config_text = "\n".join(
    f"{key} = {value!r}" for key, value in config_values.items()
) + "\n"
config_path = ROOT / "config.py"
config_path.write_text(config_text, encoding="utf-8")
config_path.chmod(0o600)
print("config.py creado dentro del runtime. No se muestran las credenciales.")

## 4. Iniciar el bot dentro de udocker

El bot se ejecuta con el Python de la imagen y el repositorio montado en `/usr/src/app`. Se omiten el entrypoint `start.sh` y el actualizador para no modificar el checkout. El runtime de udocker comparte la red del anfitrión, por lo que la celda siguiente no necesita Docker ni publicar puertos.

In [ ]:
import os
import signal
import shutil
import subprocess
import time

previous_bot = globals().get("bot_process")
if previous_bot is not None and previous_bot.poll() is None:
    raise RuntimeError("El bot ya está ejecutándose. Detén el proceso antes de volver a iniciarlo.")
UDOCKER = shutil.which("udocker")
if not UDOCKER:
    raise RuntimeError("Ejecuta primero la celda de instalación de udocker.")
BOT_LOG = ROOT / "colab_bot.log"
TUNNEL_URL_FILE = ROOT / ".colab_tunnel_url"
TUNNEL_URL_FILE.write_text("", encoding="utf-8")
bot_command = [
    UDOCKER, "run", "--user=root",
    f"--volume={ROOT}:/usr/src/app",
    "--workdir=/usr/src/app",
    "--env=PORT=8080",
    "--env=TUNNEL_URL_FILE=/usr/src/app/.colab_tunnel_url",
    "--env=WZML_DOWNLOAD_DIR=/usr/src/app/downloads",
    "--env=WZML_JDOWNLOADER_DIR=/usr/src/app/JDownloader",
    "--env=PYTHONUNBUFFERED=1",
    "--entrypoint=/wzvenv/bin/python",
    CONTAINER_NAME, "-m", "bot",
]
with BOT_LOG.open("w", encoding="utf-8") as log_file:
    bot_process = subprocess.Popen(
        bot_command,
        cwd=ROOT,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

def show_bot_log(lines=30):
    if BOT_LOG.exists():
        print("\n".join(BOT_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:]))

deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if bot_process.poll() is not None:
        show_bot_log(80)
        raise RuntimeError(f"El bot terminó con código {bot_process.returncode}; revisa el log anterior.")
    log_text = BOT_LOG.read_text(encoding="utf-8", errors="replace") if BOT_LOG.exists() else ""
    if "WZ Client(s) & Services Started" in log_text:
        break
    time.sleep(2)
else:
    print("El proceso sigue iniciando; últimas líneas:")
show_bot_log(35)
print("Bot iniciado dentro de udocker. Mantén el runtime activo.")

## 5. UI web y selector de archivos (opcional, recomendado)

Colab no expone puertos entrantes. Esta celda intenta crear un Cloudflare Quick Tunnel gratuito y temporal para el puerto 8080 del host. udocker/PRoot comparte la red del runtime, así que el túnel puede alcanzar el servidor. El enlace se pasa al bot mediante un archivo montado. Si Cloudflare no se inicia, se intenta el proxy de Colab.

In [ ]:
import os
import re
import shutil
import signal
import subprocess
import time

previous_tunnel = globals().get("tunnel_process")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    os.killpg(previous_tunnel.pid, signal.SIGTERM)
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(previous_tunnel.pid, signal.SIGKILL)
TUNNEL_LOG = ROOT / "colab_tunnel.log"
tunnel_url = ""
tunnel_process = None
if not shutil.which("cloudflared"):
    cloudflared_deb = Path("/tmp/cloudflared-linux-amd64.deb")
    download = subprocess.run(["curl", "-fL", "--retry", "2", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", "-o", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if download.returncode == 0:
        installed = subprocess.run(["dpkg", "-i", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if installed.returncode != 0:
            subprocess.run(["apt-get", "-f", "install", "-y", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(["dpkg", "-i", str(cloudflared_deb)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("cloudflared"):
    with TUNNEL_LOG.open("w", encoding="utf-8") as tunnel_log:
        tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        text = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace") if TUNNEL_LOG.exists() else ""
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if match:
            tunnel_url = match.group(0).rstrip("/")
            break
        if tunnel_process.poll() is not None:
            break
        time.sleep(2)
if not tunnel_url:
    try:
        from google.colab.output import eval_js
        tunnel_url = eval_js("google.colab.kernel.proxyPort(8080)") or ""
    except Exception as exc:
        print(f"No se pudo crear el túnel de Cloudflare ni el proxy de Colab: {exc}")
if tunnel_url:
    TUNNEL_URL_FILE.write_text(tunnel_url, encoding="utf-8")
    print("UI web del bot:", tunnel_url)
    print("El bot aplicará esta URL en unos segundos; el túnel solo vive mientras siga activo este runtime.")
else:
    print("La UI externa no quedó expuesta. El bot de Telegram puede seguir funcionando, pero los selectores web no estarán disponibles.")
    if TUNNEL_LOG.exists():
        print("\n".join(TUNNEL_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))


## Logs y apagado

Vuelve a ejecutar esta celda para consultar las últimas líneas. La celda de apagado termina los procesos del bot y del túnel; no borra `config.py`. Al terminar el runtime, Colab elimina el disco temporal.


In [ ]:
show_bot_log(60)


In [ ]:
def stop_colab_process(process, label):
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
    print(f"{label}: detenido")

stop_colab_process(globals().get("tunnel_process"), "Túnel")
stop_colab_process(globals().get("bot_process"), "Bot")
